In [ ]:
!pip install torch tensorflow ultralytics pillow matplotlib

In [ ]:
import torch
import tensorflow as tf
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import shutil
import gc

# 0. Verificação da GPU

In [ ]:
if tf.test.gpu_device_name():
    print("GPU found!")
else:
    print("GPU not found!")
    
print(torch.cuda.is_available())
print(torch.cuda.device_count())

In [ ]:
!nvidia-smi

In [ ]:
# Código para, caso a memória cache esteja cheia, esvaziar
gc.collect()
torch.cuda.empty_cache()

# 1. Configurando ambiente de treinamento

In [ ]:
%%writefile dataset.yaml

# Configuração do arquivo yaml onde tem os paths e configurações gerais que serão utilizadas

train: "/kaggle/input/datasets/nicodenico/microscopia/treino/images/Train"
test: "/kaggle/input/datasets/nicodenico/microscopia/teste/images/Test"
val: "/kaggle/input/datasets/nicodenico/microscopia/val/images"

names:
  0: mono_flakes


In [ ]:
# Código para redimensionamento das imagens e recorte das labels. Não foi utilizado.
# from sahi.slicing import slice_image

# slice_image(
#     image="imagem.png",
#     output_dir="tiles/",
#     slice_height=640,
#     slice_width=640,
#     overlap_height_ratio=0.2,
#     overlap_width_ratio=0.2
# )

# 2. Treinando

Para entender melhor os parâmetros e arquitetura da rede é possível ler a [documentação da Ultralytics](https://docs.ultralytics.com/pt/modes/train/#musgd-optimizer)

In [ ]:
model = YOLO("yolov8s-seg.pt")   # pode ser n, s, m, l, x, mas é necessário considerar underfit e overfit com base no modelo

model.train(
    data="/kaggle/working/dataset.yaml",
    imgsz=1920,
    batch= 1,
    epochs=40,
    augment=True, # Ativa vários parâmetros de aumento de dados e hiperparâmetros, possível encontrar na documentação
    degrees=15, # Gira a imagem aleatoriamente em até 15 graus
    fliplr=0.5, # Inverte a imagem horizontalmente com 50% de probabilidade
    flipud=0.5, # Inverte a imagem verticamente com 50% de probabilidade
    save_period = 5, # A cada 5 épocas salva o modelo
)

In [ ]:
# Código para, caso o treinamento seja interrompido, voltar a treinar

model = YOLO("/kaggle/working/runs/segment/train6/weights/best.pt")  # caminho do modelo

model.train(
    data="dataset.yaml",
    resume=True
)

# 3. Testando e salvando

A documentação dessa parte do código pode ser encontrada no [site do ultralytics](https://docs.ultralytics.com/pt/modes/predict/#working-with-results)

In [ ]:
model = YOLO("/kaggle/working/runs/segment/train6/weights/best.pt")  # caminho do best.pt do modelo

In [ ]:
# Predição das imagens de teste para avaliação visual e entendimento do modeo
folder_path = "/kaggle/input/datasets/nicodenico/microscopia/teste/images/Test"

model.predict(
    folder_path, 
    conf = 0.25, # threshold de detecção
    save=True, 
    visualize = True, # Ativa a visualização das características do modelo durante a inferência
    augment = True, # Ativa o aumento do tempo de teste (TTA) para previsões
    show_labels=True,  
    show_conf=True,
    show = True # Não funciona no Kaggle Notebook por configurações do ambiente, mas podendo ser modificado
)

In [ ]:
%%writefile dataset.yaml

# Mudando o teste com o val para avaliar o modelo com a biblioteca

train: "/kaggle/input/datasets/nicodenico/microscopia/treino/images/Train"
test: "/kaggle/input/datasets/nicodenico/microscopia/val/images"
val: "/kaggle/input/datasets/nicodenico/microscopia/teste/images/Test"


names:
  0: mono_flakes


In [ ]:
# Irá salvar as métricas em run/segment/val
metrics = model.val(
    data="/kaggle/working/dataset.yaml",
    imgsz=1920,
    conf=0.5,
    plots = True
)


In [ ]:
shutil.make_archive("modelo", "zip", "/kaggle/working/runs/segment/train")

shutil.make_archive("resultados", "zip", "/kaggle/working/runs/segment/predict")

shutil.make_archive("resultados_metricas", "zip", "/kaggle/input/datasets/nicodenico/val")

In [ ]:
# Deleta a pasta working para limpar a memória do Kaggle notebook (PERIGOSO!!)
!rm -rf /kaggle/working/*